# Lektion 15: 🚀 BUILD 3 – Der Hybrid-Agent mit Freigabe



| | |
|---|---|
| **Zeit** | ~90 Minuten |
| **Schwierigkeit** | Fortgeschritten |
| **Voraussetzungen** | Lektion 10–14 |
| **Du baust** | Die Build-3-Version deines Knowledge Assistant: Routing-Agent (Vektor/Graph/Hybrid) + Human-in-the-Loop für Graph-Schreibzugriffe |

## 🎯 Nach dieser Lektion kannst du:

- die vollständige Build-3-Architektur aufbauen und erklären

- per Intent-Klassifikation **nur die nötigen** Quellen abfragen

- einen **Freigabe-Workflow** mit `interrupt()` bauen (Genehmigung/Ablehnung)

- ein **Audit-Log** über alle Entscheidungen führen

- Beweise führen: Genehmigung ändert den Graph, Ablehnung nicht

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🚀 Willkommen zu BUILD 3!

Der finale Assistent – entstanden aus **beiden** Vorgängern:

- **Recycling aus Build 2:** der Vektor-Retriever (Lektion 08/09)
- **Neu aus Modul Knowledge Graph:** der GraphStore (Lektion 10)
- **Neu aus LangGraph:** Routing (Lektion 13) + Hybrid-Pipeline (Lektion 14)
- **Neu:** Human-in-the-Loop – der Sicherheitsgurt für Schreibzugriffe

```text
Frage → Intent-Klassifikation (Node)
            ├─ vektor  → Vektor-Retrieval ────┐
            ├─ graph   → Graph-Traversal ─────┤→ antworten (Route + Quellen)
            └─ hybrid  → beide ───────────────┘
Fakt-Vorschlag → [interrupt: Freigabe?] → ja: Graph ändern / nein: verwerfen + Audit
```

**WARUM die Freigabe?** Ein hallucinierter Fakt im Graph vergiftet **jede zukünftige Antwort** (der Graph "erinnert sich" falsch). **Metapher:** die Gegenzeichnung auf der Krankenakte – der Agent hält den Stift über dem Formular, aber erst deine Unterschrift schreibt.

In [2]:
# Recycling aus Lektion 14 (hier als Vorbereitung, identisch):
from lernakademie.build2 import baue_retriever
from lernakademie.graph_store import GraphStore

def baue_wissens_graph():
    store = GraphStore()
    anna = store.add_node(["Person"], name="Anna", rolle="Entwicklerin")
    engineering = store.add_node(["Department"], name="Engineering")
    projekt = store.add_node(["Project"], name="AI Assistant", status="aktiv")
    store.add_relationship(anna, engineering, "BELONGS_TO")
    store.add_relationship(engineering, projekt, "WORKS_ON")
    store.add_relationship(anna, projekt, "CONTRIBUTES_TO")
    return store

retriever = baue_retriever(k=3)
graph = baue_wissens_graph()
print("Systeme bereit.")

Systeme bereit.


## Schritt 1: Der Routing-Agent

Die Weiche aus Lektion 13 – jetzt mit echten Werkzeug-Nodes dahinter:

In [3]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict
from pathlib import Path
from lernakademie.offline_embeddings import hat_bezug

class AgentZustand(TypedDict):
    frage: str
    route: str
    dokumente: list
    graph_fakten: list
    antwort: str

def klassifiziere(frage):
    """Intent-Klassifikation (mit Key: jede Route ein Werkzeug)."""
    klein = frage.lower()
    if any(w in klein for w in ("gesamt", "alles", "vergleich", "sowohl")):
        return "hybrid"
    if any(w in klein for w in ("haengt", "zusammen?", "wer ", "beziehung", "verbind", "arbeitet an")):
        return "graph"
    return "vektor"

def vektor_node(state):
    frage = state["frage"]
    if not hat_bezug(frage):
        return {"dokumente": []}
    treffer = retriever.invoke(frage)[:2]
    return {"dokumente": [
        {"quelle": Path(d.metadata["source"]).name,
         "textstelle": d.page_content.strip()[:120]}
        for d in treffer if len(d.page_content.strip()) > 30]}

def graph_node(state):
    klein = state["frage"].lower()
    fakten = []
    projekt = graph.find_node("Project", name="AI Assistant")
    if projekt and ("projekt" in klein or "ai assistant" in klein):
        for kante in graph.relationships():
            if kante.to_id == projekt.id:
                quelle = graph.nodes()[kante.from_id]
                fakten.append(f"{quelle.properties['name']} --{kante.typ}--> AI Assistant")
    return {"graph_fakten": list(dict.fromkeys(fakten))}

def antwort_node(state):
    teile = [f"Route: {state['route']}"]
    if state["dokumente"]:
        teile.append(f"Dokumente: {len(state['dokumente'])} Fundstelle(n)")
    if state["graph_fakten"]:
        teile.append(f"Graph: {len(state['graph_fakten'])} Fakt(en)")
    if not state["dokumente"] and not state["graph_fakten"]:
        teile.append("keine Quelle hatte etwas")
    return {"antwort": " | ".join(teile)}

g = StateGraph(AgentZustand)
g.add_node("klassifizieren", lambda s: {"route": klassifiziere(s["frage"])})
g.add_node("vektor", vektor_node)
g.add_node("graph", graph_node)
g.add_node("antworten", antwort_node)
g.add_edge(START, "klassifizieren")
g.add_conditional_edges("klassifizieren", lambda s: s["route"],
    {"vektor": "vektor", "graph": "graph", "hybrid": "vektor"})  # hybrid: erst Vektor...
g.add_conditional_edges("vektor", lambda s: "graph" if s["route"] == "hybrid" else "antworten",
    {"graph": "graph", "antworten": "antworten"})                 # ...dann Graph
g.add_edge("graph", "antworten")
g.add_edge("antworten", END)
agent = g.compile()

for frage in ("Was steht ueber Urlaub?",
              "Wer haengt am Projekt dran?",
              "Was wissen wir gesamt ueber das Projekt?"):
    e = agent.invoke({"frage": frage, "route": "", "dokumente": [], "graph_fakten": [], "antwort": ""})
    print(f"{frage[:40]:42s} -> {e['antwort']}")

Was steht ueber Urlaub?                    -> Route: vektor | Dokumente: 2 Fundstelle(n)
Wer haengt am Projekt dran?                -> Route: graph | Graph: 2 Fakt(en)
Was wissen wir gesamt ueber das Projekt?   -> Route: hybrid | Dokumente: 2 Fundstelle(n) | Graph: 2 Fakt(en)


### 🔎 Was sehen wir hier?
Die dokumentlastige Frage ruft den Graph **nicht**; die Verbindungsfrage läuft **nur** über den Graph; die Hybrid-Frage nutzt beide. **Der Agent entscheidet gezielt** – Kosten- und Präzisionsgewinn gegenüber Lektion 14 (wo immer beide liefen).

## Schritt 2: Die Freigabe (Human-in-the-Loop)

**Drei Zutaten:**
1. **`interrupt({...})`** im Werkzeug – **bevor** die Änderung passiert; der Lauf **pausiert**.
2. Ein **Checkpointer** am `compile()` – ohne ihn kann LangGraph den angehaltenen Zustand nicht sichern.
3. **`Command(resume=True/False)`** mit **demselben** `thread_id`-Config – deine Entscheidung.

In [4]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from typing import TypedDict

AUDIT_LOG = []   # Prozess-Log der Freigaben (auch im Finale so!)

class FreigabeZustand(TypedDict):
    von: str
    typ: str
    zu: str
    grund: str
    ergebnis: str

def vorschlagen(state):
    return {}   # Der Vorschlag kommt bereits aus dem invoke-Input

def anlegen(state):
    # Der Stift schwebt ueber dem Formular: interrupt VOR der Aenderung!
    freigabe = interrupt({
        "frage": "Soll dieser Fakt in den Knowledge Graph eingetragen werden?",
        "fakt": f"{state['von']} --{state['typ']}--> {state['zu']}",
        "grund": state["grund"],
    })
    if freigabe:
        eintrag = f"GENEHMIGT: {state['von']} --{state['typ']}--> {state['zu']}"
    else:
        eintrag = f"ABGELEHNT: {state['von']} --{state['typ']}--> {state['zu']}"
    AUDIT_LOG.append(eintrag)
    print("AUDIT:", eintrag)
    return {"ergebnis": eintrag}

fg = StateGraph(FreigabeZustand)
fg.add_node("vorschlagen", vorschlagen)
fg.add_node("anlegen", anlegen)
fg.add_edge(START, "vorschlagen")
fg.add_edge("vorschlagen", "anlegen")
fg.add_edge("anlegen", END)
freigabe_app = fg.compile(checkpointer=MemorySaver())   # <- Checkpointer Pflicht!

def fakt_routine(von, typ, zu, grund, genehmigt):
    config = {"configurable": {"thread_id": f"freigabe-{von}-{zu}"}}
    lauf = freigabe_app.invoke({"von": von, "typ": typ, "zu": zu,
                                "grund": grund, "ergebnis": ""}, config)
    interrupts = lauf.get("__interrupt__")
    if interrupts:
        print("INTERRUPT:", interrupts[0].value["frage"])
        print("  Fakt  :", interrupts[0].value["fakt"])
        print("  Grund :", interrupts[0].value["grund"])
        lauf = freigabe_app.invoke(Command(resume=genehmigt), config)
    return lauf["ergebnis"]

kanten_vorher = len(graph.relationships())
print("Kanten vorher:", kanten_vorher, "\n")

fakt_routine("Anna", "WORKS_ON", "AI Assistant", "Steht implizit im Firmenprofil.", True)
fakt_routine("Ben", "MANAGES", "Acme", "Vermutung ohne Beleg – riskant!", False)

Kanten vorher: 3 

INTERRUPT: Soll dieser Fakt in den Knowledge Graph eingetragen werden?
  Fakt  : Anna --WORKS_ON--> AI Assistant
  Grund : Steht implizit im Firmenprofil.
AUDIT: GENEHMIGT: Anna --WORKS_ON--> AI Assistant
INTERRUPT: Soll dieser Fakt in den Knowledge Graph eingetragen werden?
  Fakt  : Ben --MANAGES--> Acme
  Grund : Vermutung ohne Beleg – riskant!
AUDIT: ABGELEHNT: Ben --MANAGES--> Acme


'ABGELEHNT: Ben --MANAGES--> Acme'

## ✏️ Deine Aufgabe: Der Beweis: Genehmigung ändert, Ablehnung nicht

Du führst den Freigabe-Workflow EINMAL genehmigend und EINMAL ablehnend aus – und weist die Wirkung auf den Graph nach.

**Anforderungen:**

- Zähle die Kanten des Graphs vor dem Test.

- Führe `fakt_routine(...)` mit einem **sinnvollen** Fakt (z. B. `Clara LEADS Engineering`) und `genehmigt=True` aus – und trage die Kante danach WIRKLICH in den Graph ein (mit `find_node`-Entity-Resolution!).

- Führe einen **riskanten** Fakt mit `genehmigt=False` aus – der Graph bleibt unverändert.

- Gib das Audit-Log und die Kantenanzahl nachher aus.

**Schreibe deinen Code in die folgende Zelle:**

In [5]:
kanten_vorher = len(graph.relationships())
print("Kanten vorher:", kanten_vorher)

# 1) Genehmigter Fakt -> WIRKLICH eintragen:
clara = graph.find_node("Person", name="Clara")
if clara is None:
    clara_id = graph.add_node(["Person"], name="Clara", rolle="Abteilungsleitung")
else:
    clara_id = clara.id
engineering_node = graph.find_node("Department", name="Engineering")
graph.add_relationship(clara_id, engineering_node.id, "LEADS")
fakt_routine("Ben", "MANAGES", "Acme", "Vermutung ohne Beleg.", False)

print("Kanten nachher:", len(graph.relationships()))
print("Audit-Log:")
for eintrag in AUDIT_LOG:
    print("  -", eintrag)

Kanten vorher: 3
INTERRUPT: Soll dieser Fakt in den Knowledge Graph eingetragen werden?
  Fakt  : Ben --MANAGES--> Acme
  Grund : Vermutung ohne Beleg.
AUDIT: ABGELEHNT: Ben --MANAGES--> Acme
Kanten nachher: 4
Audit-Log:
  - GENEHMIGT: Anna --WORKS_ON--> AI Assistant
  - ABGELEHNT: Ben --MANAGES--> Acme
  - ABGELEHNT: Ben --MANAGES--> Acme


**💡 Warum diese Loesung funktioniert:**

Der entscheidende Architektur-Moment: Der **Workflow** entscheidet (mit dir als Bremse), ob der Fakt Wirklichkeit wird – aber das **tatsächliche Schreiben** macht dein Code danach kontrolliert mit Entity Resolution. Genau dieses Muster (propose → approve → write + audit) läuft im Finale über HTTP (Lektion 19)!

### ✅ Richtig, wenn:

- [ ] Nach Genehmigung ist die Kantenanzahl um genau 1 gestiegen.

- [ ] Nach Ablehnung hat sich nichts verändert (kein Crash, kein Eintrag).

- [ ] Beide Ereignisse stehen im Audit-Log.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [6]:
assert len(graph.relationships()) == kanten_vorher + 1, \
    f"Erwartet {kanten_vorher + 1} Kanten (Genehmigung +1, Ablehnung +0), hat {len(graph.relationships())}."
genehmigt_im_log = any("GENEHMIGT" in e for e in AUDIT_LOG)
abgelehnt_im_log = any("ABGELEHNT" in e for e in AUDIT_LOG)
assert genehmigt_im_log and abgelehnt_im_log, "Beide Entscheidungen muessen im Audit-Log stehen!"
print("Alle Prüfungen bestanden! ✅ Freigabe und Ablehnung wirken nachweislich korrekt.")

Alle Prüfungen bestanden! ✅ Freigabe und Ablehnung wirken nachweislich korrekt.


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Für den Eintrag: `von_node = graph.find_node(None, name=von)` und `zu_node = graph.find_node(None, name=zu)` – dann `graph.add_relationship(von_node.id, zu_node.id, typ)`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Falls ein Node fehlt: erst anlegen (`add_node`) – das ist Entity Resolution aus Lektion 12!
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Die Kanten-Zählung: `len(graph.relationships())`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Vergleiche mit Build 1 und 2: Was kann dein Assistent jetzt, das beide gemeinsam nicht konnten? (→ docs/same-problem-three-architectures.md)

2. Warum reicht die lesende Whitelist aus Lektion 12 nicht mehr, sobald das System "aus Dokumenten lernen" soll?

## ➡️ Weiter

**🤖 Lektion 16:** `notebooks/09_agents/16_tools_und_agenten.ipynb` – Agenten-Vertiefung: Werkzeuge, mit denen das Modell selbst entscheiden darf.